# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')  # pulled from Colab Secrets, never printed or stored in the file

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("Connected.")

Connected.


In [7]:
import os

# Clone only if not already present (safe to re-run without erroring)
if not os.path.exists('/content/ML-intern-starter'):
    !git clone https://github.com/Khuld13/ML-intern-starter.git

%cd /content/ML-intern-starter
!pwd

# --- HF connection ---
from google.colab import userdata
import duckdb
import pandas as pd

hf_token = userdata.get('HF_TOKEN')  # pulled from Colab Secrets, never printed or stored in the file

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("Connected.")

Cloning into 'ML-intern-starter'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 161 (delta 67), reused 88 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 1.89 MiB | 9.87 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/ML-intern-starter
/content/ML-intern-starter
Connected.


In [8]:
con.sql("SHOW TABLES").df()

,name


In [9]:
con.sql("""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""")
con.sql("""
    CREATE OR REPLACE VIEW fact_march AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""")
print("Views ready.")

Views ready.


In [10]:
con.sql("SHOW TABLES").df()

,name
0,dim_content
1,fact_march


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Distributions of key fields
# gsc_impressions, avg_position -> fact_march (daily performance, dev view)
# days_since_update -> dim_content (July 2026 snapshot)

dist_query = """
SELECT 'gsc_impressions' AS field,
    MIN(gsc_impressions) AS min_val,
    approx_quantile(gsc_impressions, 0.25) AS p25,
    approx_quantile(gsc_impressions, 0.50) AS median,
    approx_quantile(gsc_impressions, 0.75) AS p75,
    approx_quantile(gsc_impressions, 0.95) AS p95,
    approx_quantile(gsc_impressions, 0.99) AS p99,
    MAX(gsc_impressions) AS max_val,
    AVG(gsc_impressions) AS mean_val
FROM fact_march
WHERE gsc_data_available = TRUE

UNION ALL

SELECT 'avg_position',
    MIN(avg_position),
    approx_quantile(avg_position, 0.25),
    approx_quantile(avg_position, 0.50),
    approx_quantile(avg_position, 0.75),
    approx_quantile(avg_position, 0.95),
    approx_quantile(avg_position, 0.99),
    MAX(avg_position),
    AVG(avg_position)
FROM fact_march
WHERE gsc_data_available = TRUE

UNION ALL

SELECT 'days_since_update',
    MIN(days_since_update),
    approx_quantile(days_since_update, 0.25),
    approx_quantile(days_since_update, 0.50),
    approx_quantile(days_since_update, 0.75),
    approx_quantile(days_since_update, 0.95),
    approx_quantile(days_since_update, 0.99),
    MAX(days_since_update),
    AVG(days_since_update)
FROM dim_content
"""

con.sql(dist_query).df()

BinderException: Binder Error: Referenced column "avg_position" not found in FROM clause!
Candidate bindings: "gsc_avg_position", "gsc_sum_position", "ai_copilot", "gsc_impressions", "ga4_sessions"

In [12]:
# Get ref_date first (max report_date in March partition)
ref_date = con.sql("SELECT MAX(report_date) AS d FROM fact_march").df()['d'][0]
print("Reference date:", ref_date)

dist_query = f"""
SELECT 'gsc_impressions' AS field,
    MIN(gsc_impressions) AS min_val,
    approx_quantile(gsc_impressions, 0.25) AS p25,
    approx_quantile(gsc_impressions, 0.50) AS median,
    approx_quantile(gsc_impressions, 0.75) AS p75,
    approx_quantile(gsc_impressions, 0.95) AS p95,
    approx_quantile(gsc_impressions, 0.99) AS p99,
    MAX(gsc_impressions) AS max_val,
    AVG(gsc_impressions) AS mean_val
FROM fact_march
WHERE gsc_data_available = TRUE

UNION ALL

SELECT 'gsc_avg_position',
    MIN(gsc_avg_position),
    approx_quantile(gsc_avg_position, 0.25),
    approx_quantile(gsc_avg_position, 0.50),
    approx_quantile(gsc_avg_position, 0.75),
    approx_quantile(gsc_avg_position, 0.95),
    approx_quantile(gsc_avg_position, 0.99),
    MAX(gsc_avg_position),
    AVG(gsc_avg_position)
FROM fact_march
WHERE gsc_data_available = TRUE

UNION ALL

SELECT 'days_since_update',
    MIN(DATE '{ref_date}' - content_updated_date),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.25),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.50),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.75),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.95),
    approx_quantile(DATE '{ref_date}' - content_updated_date, 0.99),
    MAX(DATE '{ref_date}' - content_updated_date),
    AVG(DATE '{ref_date}' - content_updated_date)
FROM dim_content
"""

con.sql(dist_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Reference date: 2026-03-31 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,field,min_val,p25,median,p75,p95,p99,max_val,mean_val
0,gsc_impressions,1.0,4.000000,16.000000,62.000000,337.000000,949.000000,40084.0,77.721642
1,gsc_avg_position,0.0,3.734549,7.473027,20.202337,62.651515,88.619121,498.0,15.826651
2,days_since_update,-97.0,-62.000000,-50.000000,32.000000,492.000000,509.000000,519.0,34.148043


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.